# Chapter 11 — SOLUTIONS: Basic RL Algorithms

**Instructor file — share only after exercise session.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
np.random.seed(42)

# Grid environment
GRID = np.array([[3,0,0,0],[0,1,0,1],[0,0,0,1],[1,0,0,2]])
ROWS, COLS = 4, 4
N_STATES, N_ACTIONS = 16, 4
MOVE_DR = {0:(0,-1),1:(1,0),2:(0,1),3:(-1,0)}
REWARDS = {0:-0.01,1:-1.0,2:1.0,3:-0.01}
_state = [0]

def env_reset():
    _state[0] = 0
    return 0

def env_step(action):
    r,c = _state[0]//COLS, _state[0]%COLS
    dr,dc = MOVE_DR[action]
    nr,nc = max(0,min(ROWS-1,r+dr)), max(0,min(COLS-1,c+dc))
    ns = nr*COLS+nc
    ct = GRID[nr,nc]
    reward = REWARDS[ct]
    done = ct in [1,2]
    _state[0] = ns
    return ns, reward, done

print('Setup complete!')

## Task 1 Solution: Initialize the Q-Table

In [ ]:
Q = np.zeros((N_STATES, N_ACTIONS))
print('Q-table shape:', Q.shape)
print('Initial Q values (first 4 states):')
print(Q[:4])

## Task 2 Solution: ε-Greedy Action Selection

In [ ]:
def choose_action(state, Q, epsilon):
    if np.random.random() < epsilon:
        return np.random.randint(N_ACTIONS)   # explore: random action
    else:
        return np.argmax(Q[state])            # exploit: best known action

Q_test = np.array([[0.1, 0.8, 0.3, 0.2]] * N_STATES)
print('With epsilon=0 (always exploit):', choose_action(0, Q_test, epsilon=0))
print('→ Should be 1 (Down) — highest Q-value')
print('With epsilon=1 (always explore):', [choose_action(0, Q_test, epsilon=1) for _ in range(8)])
print('→ Should be random (0-3)')

## Task 3 Solution: The Bellman Update

In [ ]:
def q_update(Q, state, action, reward, next_state, done, alpha=0.8, gamma=0.95):
    # If episode is done, no future Q value
    future_q = 0 if done else gamma * np.max(Q[next_state])
    td_target = reward + future_q
    td_error  = td_target - Q[state, action]
    Q[state, action] += alpha * td_error

# Test
Q_test = np.zeros((N_STATES, N_ACTIONS))
q_update(Q_test, state=0, action=2, reward=1.0, next_state=15, done=True, alpha=0.8, gamma=0.95)
print(f'Q[0, 2] = {Q_test[0, 2]:.4f}  (expected: 0.8000)')
print('= 0 + 0.8 * (1.0 + 0 - 0) = 0.8 ✓')

## Task 4 Solution: Full Training Loop

In [ ]:
ALPHA, GAMMA = 0.8, 0.95
EPSILON = 1.0
N_EPISODES, MAX_STEPS = 3000, 100

Q = np.zeros((N_STATES, N_ACTIONS))
epsilon = EPSILON
success_history = []

for episode in range(N_EPISODES):
    state = env_reset()
    success = False
    
    for step in range(MAX_STEPS):
        action = choose_action(state, Q, epsilon)
        next_state, reward, done = env_step(action)
        q_update(Q, state, action, reward, next_state, done, ALPHA, GAMMA)
        state = next_state
        
        if done:
            success = (reward > 0)  # goal reached
            break
    
    success_history.append(float(success))
    epsilon = max(epsilon * 0.995, 0.01)

print(f'Training complete! Final epsilon: {epsilon:.4f}')
print(f'Success rate (last 500 eps): {np.mean(success_history[-500:]):.1%}')

In [ ]:
# Rolling success rate plot
window = 100
rolling = [np.mean(success_history[max(0,i-window):i+1]) for i in range(len(success_history))]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rolling, color='#2ecc71', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel(f'Success Rate (rolling {window})')
ax.set_title('Q-Learning: Agent Learns to Navigate the Grid!', fontsize=13)
ax.axhline(np.mean(success_history[-500:]), color='red', linestyle='--', alpha=0.7,
            label=f'Final: {np.mean(success_history[-500:]):.1%}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('The agent improves from near-0% to a high success rate!')
print('This is Q-Learning working: trial, error, Bellman update, repeat.')

## Bonus Solution: Visualize the Learned Policy

In [ ]:
action_arrows = {0:'←', 1:'↓', 2:'→', 3:'↑'}
best_actions  = np.argmax(Q, axis=1)
special = {5:'H', 7:'H', 11:'H', 12:'H', 15:'G', 0:'S'}

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.grid(True, linewidth=1.5, color='#bdc3c7')

for s in range(N_STATES):
    r, c = s // COLS, s % COLS
    label = special.get(s, action_arrows[best_actions[s]])
    color = '#e74c3c' if s in [5,7,11,12] else '#2ecc71' if s == 15 else '#3498db' if s == 0 else '#2c3e50'
    ax.text(c + 0.5, 3.5 - r, label, ha='center', va='center',
            fontsize=20, fontweight='bold', color=color)

ax.set_title('Learned Policy (Q-Learning)\nS=Start, H=Hole, G=Goal, Arrows=Best Action', fontsize=11)
plt.tight_layout()
plt.show()

## Bonus Solution: Implement SARSA

In [ ]:
ALPHA, GAMMA = 0.8, 0.95
N_EPISODES, MAX_STEPS = 3000, 100

Q_sarsa = np.zeros((N_STATES, N_ACTIONS))
epsilon_sarsa = 1.0
success_history_sarsa = []

for episode in range(N_EPISODES):
    state = env_reset()
    action = choose_action(state, Q_sarsa, epsilon_sarsa)
    success = False

    for step in range(MAX_STEPS):
        next_state, reward, done = env_step(action)
        next_action = choose_action(next_state, Q_sarsa, epsilon_sarsa)

        # SARSA update: use Q[next_state, next_action] — not max!
        future_q = 0 if done else GAMMA * Q_sarsa[next_state, next_action]
        td_error = reward + future_q - Q_sarsa[state, action]
        Q_sarsa[state, action] += ALPHA * td_error

        state, action = next_state, next_action
        if done:
            success = (reward > 0)
            break

    success_history_sarsa.append(float(success))
    epsilon_sarsa = max(epsilon_sarsa * 0.995, 0.01)

print(f'SARSA Success rate (last 500 eps): {np.mean(success_history_sarsa[-500:]):.1%}')

# Compare Q-Learning vs SARSA
window = 100
rolling_ql   = [np.mean(success_history[max(0,i-window):i+1])     for i in range(N_EPISODES)]
rolling_sarsa= [np.mean(success_history_sarsa[max(0,i-window):i+1]) for i in range(N_EPISODES)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rolling_ql,    color='#2ecc71', linewidth=2, label='Q-Learning')
ax.plot(rolling_sarsa, color='#e74c3c', linewidth=2, label='SARSA', linestyle='--')
ax.set_xlabel('Episode')
ax.set_ylabel(f'Success Rate (rolling {window})')
ax.set_title('Q-Learning vs SARSA: Learning Curves', fontsize=13)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print('\nKey difference: SARSA is on-policy — it updates using the action it actually takes.')
print('Q-Learning is off-policy — it updates assuming the optimal action.')
print('On this simple grid, both converge similarly, but SARSA tends to be'
      ' safer in environments with risky actions during exploration.')